In [10]:
import os
import cv2
import shutil
import random
import numpy as np
from pathlib import Path
from tqdm import tqdm

In [11]:
NOTEBOOK_DIR = Path.cwd()

# Project root (VisionXM)
PROJECT_ROOT = NOTEBOOK_DIR.parent

# MVTec dataset
DATA_ROOT = PROJECT_ROOT / "data"

TRAIN_GOOD = DATA_ROOT / "train" / "good"
TEST_ROOT = DATA_ROOT / "test"

# YOLO dataset will be created at:
# VisionXM/yolo_dataset
YOLO_ROOT = PROJECT_ROOT / "yolo_dataset"

TRAIN_IMG = YOLO_ROOT / "images" / "train"
VAL_IMG = YOLO_ROOT / "images" / "val"

TRAIN_LABEL = YOLO_ROOT / "labels" / "train"
VAL_LABEL = YOLO_ROOT / "labels" / "val"

for folder in [
    TRAIN_IMG,
    VAL_IMG,
    TRAIN_LABEL,
    VAL_LABEL,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project Root :", PROJECT_ROOT)
print("Data Root    :", DATA_ROOT)
print("YOLO Dataset :", YOLO_ROOT)

Project Root : d:\Projects\VisionXM
Data Root    : d:\Projects\VisionXM\data
YOLO Dataset : d:\Projects\VisionXM\yolo_dataset


In [12]:
def get_bbox(image):

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    _, thresh = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    contours, _ = cv2.findContours(
        thresh,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours) == 0:
        return None

    largest = max(contours, key=cv2.contourArea)

    x, y, w, h = cv2.boundingRect(largest)

    return x, y, w, h

In [13]:
def bbox_to_yolo(x, y, w, h, img_w, img_h):

    x_center = (x + w / 2) / img_w
    y_center = (y + h / 2) / img_h

    w /= img_w
    h /= img_h

    return x_center, y_center, w, h

In [14]:
all_images = []

for file in TRAIN_GOOD.glob("*.png"):
    all_images.append(file)

for folder in TEST_ROOT.iterdir():

    if folder.is_dir():

        for file in folder.glob("*.png"):

            all_images.append(file)

print(len(all_images))

480


In [15]:
random.seed(42)

random.shuffle(all_images)

split = int(len(all_images) * 0.8)

train_images = all_images[:split]
val_images = all_images[split:]

print("Training images: ", len(train_images))
print("Validation images: ", len(val_images))

Training images:  384
Validation images:  96


In [16]:
def process_dataset(image_list, image_dir, label_dir):

    copied = 0

    for idx, img_path in enumerate(tqdm(image_list)):

        img = cv2.imread(str(img_path))

        if img is None:
            print("Couldn't read:", img_path)
            continue

        h, w = img.shape[:2]

        bbox = get_bbox(img)

        if bbox is None:
            print("No bbox:", img_path)
            continue

        x, y, bw, bh = bbox

        xc, yc, bw, bh = bbox_to_yolo(x, y, bw, bh, w, h)

        folder = img_path.parent.name
        filename = f"{folder}_{img_path.stem}"

        img_dest = image_dir / f"{filename}.png"
        label_dest = label_dir / f"{filename}.txt"

        # Save the image instead of copying it
        success = cv2.imwrite(str(img_dest), img)

        if not success:
            print(f"Failed to save {img_dest}")
            continue

        with open(label_dest, "w") as f:
            f.write(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

        copied += 1

    print(f"\nCopied {copied} images.")

In [17]:
process_dataset(train_images, TRAIN_IMG, TRAIN_LABEL)

  3%|▎         | 10/384 [00:00<00:04, 87.10it/s]

Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\thread_side_018.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_245.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\thread_top_007.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_059.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_267.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_092.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_215.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\manipulated_front_020.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\scratch_neck_016.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_169.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_244.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_314.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\scra

  7%|▋         | 28/384 [00:00<00:04, 80.64it/s]

Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_095.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_109.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_263.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_187.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_143.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\scratch_head_006.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_007.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_286.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_021.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_207.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_088.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_268.png
Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_253.png
Failed to save d:

 12%|█▏        | 47/384 [00:00<00:04, 82.54it/s]

...

Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_006.png


 71%|███████▏  | 274/384 [00:06<00:03, 33.07it/s]

Failed to save d:\Projects\VisionXM\yolo_dataset\images\train\good_021.png


100%|██████████| 384/384 [00:10<00:00, 37.21it/s]


Copied 274 images.


In [18]:
yaml_text = f"""
path: {YOLO_ROOT.resolve()}

train: images/train
val: images/val

names:
  0: screw
"""

with open(YOLO_ROOT / "data.yaml", "w") as f:
    f.write(yaml_text)

print("Created:", YOLO_ROOT / "data.yaml")

Created: d:\Projects\VisionXM\yolo_dataset\data.yaml
